# Experiment Notebook

Use this notebook to inspect datasets and try ideas without changing the Streamlit app files.

## Distinctive Words by Article Cluster

This section uses c-TF-IDF to identify the top distinctive words for each article cluster.

In [71]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

In [72]:
df = pd.read_parquet('iran_war_media_framing_scores_clustered.parquet', engine='pyarrow')
df[['article_cluster', 'media_name', 'title', 'article_text']].head()

,article_cluster,media_name,title,article_text
0,4,airforcetimes.com,Democrats slam Hegseth for comments on first U...,Democrats are condemning Defense Secretary Pet...
1,1,airforcetimes.com,"Despite air dominance, US ‘can’t stop everythi...",Defense Secretary Pete Hegseth acknowledged on...
2,1,airforcetimes.com,A-10 Warthogs target Iranian fast-attack craft...,U.S. Air Force A-10 Thunderbolt II attack airc...
3,1,airforcetimes.com,US has destroyed entire class of Iranian warsh...,Since the United States and Israel began their...
4,0,airforcetimes.com,"Iran to face ‘most intense day of strikes,’ He...",U.S. Defense Secretary Pete Hegseth warned tha...


Combine all articles in each cluster into one document.

In [73]:
cluster_docs = (
    df.dropna(subset=['article_text', 'article_cluster'])
    .assign(article_cluster=lambda data: data['article_cluster'].astype(int))
    .groupby('article_cluster')['article_text']
    .apply(lambda texts: ' '.join(texts.astype(str)))
    .reset_index(name='cluster_text')
)

cluster_docs

,article_cluster,cluster_text
0,0,U.S. Defense Secretary Pete Hegseth warned tha...
1,1,Defense Secretary Pete Hegseth acknowledged on...
2,2,Russia has provided Iran with information that...
3,3,Some states are stepping in to help drivers ea...
4,4,Democrats are condemning Defense Secretary Pet...


Compute c-TF-IDF scores for unigrams.

In [74]:
domain_stop_words = {
    'said', 'says', 'told', 'according', 'reported', 'reports',
    'news', 'article', 'media', 'people',
    'day', 'days', 'week', 'weeks', 'month', 'months', 'year', 'years', 'time',
    'new', 'just', 'like', 'including', 'called', 'statement', 'percent',
    'reuters', 'ap',
    'des', 'les',
    'khork', 'cody', 'nicole'
}

blocked_terms = {'khork', 'cody', 'nicole'}

blocked_phrases = {
    'john yang', 'nick schifrin', 'hegseth caine', 'trump hegseth',
    'cody khork', 'capt cody', 'class nicole', 'sgt class'
}

stop_words = list(ENGLISH_STOP_WORDS.union(domain_stop_words))

vectorizer = CountVectorizer(
    lowercase=True,
    strip_accents='unicode',
    stop_words=stop_words,
    ngram_range=(1, 1),
    min_df=1,
    max_df=0.7,
    token_pattern=r'(?u)\b[a-zA-Z]{3,}\b'
)

count_matrix = vectorizer.fit_transform(cluster_docs['cluster_text'])
terms = vectorizer.get_feature_names_out()

term_counts = count_matrix.toarray().astype(float)
cluster_lengths = term_counts.sum(axis=1, keepdims=True)
tf = np.divide(term_counts, cluster_lengths, where=cluster_lengths != 0)

term_presence = (term_counts > 0).sum(axis=0)
idf = np.log((1 + len(cluster_docs)) / (1 + term_presence)) + 1
ctfidf = tf * idf

ctfidf.shape

/var/folders/bw/1dh_ldh12pz9s_1_677m4czw0000gn/T/ipykernel_79786/2231001785.py:35: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  tf = np.divide(term_counts, cluster_lengths, where=cluster_lengths != 0)


(5, 23683)

Select the top 15 distinctive words per cluster.

In [75]:
top_n = 10
cluster_names = {
    0: 'Mainstream Consensus',
    1: 'Critical Opposition',
    2: 'Humanitarian Diplomacy',
    3: 'Economic Fallout',
    4: 'Militarized Blame'
}

rows = []

for row_idx, cluster_id in enumerate(cluster_docs['article_cluster']):
    top_indices = []

    for term_idx in np.argsort(ctfidf[row_idx])[::-1]:
        if terms[term_idx] in blocked_terms:
            continue

        top_indices.append(term_idx)

        if len(top_indices) == top_n:
            break

    for term_idx in top_indices:
        rows.append({
            'cluster': f'Cluster {cluster_id}: {cluster_names.get(cluster_id, "Unnamed")}',
            'term': terms[term_idx],
            'ctfidf_score': ctfidf[row_idx, term_idx]
        })

top_terms = pd.DataFrame(rows)
top_terms.head(20)

,cluster,term,ctfidf_score
0,Cluster 0: Mainstream Consensus,sirens,0.006679
1,Cluster 0: Mainstream Consensus,pars,0.006071
2,Cluster 0: Mainstream Consensus,brent,0.004705
3,Cluster 0: Mainstream Consensus,lng,0.004554
4,Cluster 0: Mainstream Consensus,magdy,0.004389
5,Cluster 0: Mainstream Consensus,barrages,0.003474
6,Cluster 0: Mainstream Consensus,qatarenergy,0.003109
7,Cluster 0: Mainstream Consensus,irbil,0.003036
8,Cluster 0: Mainstream Consensus,chokepoint,0.003017
9,Cluster 0: Mainstream Consensus,laffan,0.003017


Display a faceted bar chart of the most distinctive words in each cluster.

In [76]:
cluster_colors = {
    'Cluster 0: Mainstream Consensus': '#4E79A7',
    'Cluster 1: Critical Opposition': '#A0CBE8',
    'Cluster 2: Humanitarian Diplomacy': '#76B7B2',
    'Cluster 3: Economic Fallout': '#59A14F',
    'Cluster 4: Militarized Blame': '#8CD17D'
}

def make_cluster_word_chart(cluster_label, data):
    chart_data = data[data['cluster'] == cluster_label].sort_values('ctfidf_score')
    short_title = cluster_label.split(': ', 1)[-1]

    fig = px.bar(
        chart_data,
        x='ctfidf_score',
        y='term',
        orientation='h',
        title=short_title,
        labels={
            'ctfidf_score': 'c-TF-IDF Score',
            'term': ''
        },
        height=360
    )

    fig.update_traces(
        marker_color=cluster_colors[cluster_label],
        hovertemplate='%{y}: %{x:.4f}<extra></extra>'
    )

    fig.update_layout(
        plot_bgcolor='#f7fafc',
        paper_bgcolor='white',
        showlegend=False,
        bargap=0.32,
        margin={'l': 150, 'r': 35, 't': 60, 'b': 50},
        title={'x': 0.02, 'xanchor': 'left', 'font': {'size': 16, 'color': '#2f4a5f'}},
        font={'color': '#263746'}
    )

    fig.update_xaxes(showgrid=True, gridcolor='#e6edf3', zeroline=False)
    fig.update_yaxes(categoryorder='array', categoryarray=chart_data['term'].tolist())

    return fig

cluster_figures = {
    cluster_label: make_cluster_word_chart(cluster_label, top_terms)
    for cluster_label in cluster_colors
}

for fig in cluster_figures.values():
    fig.show()

## Distinctive Bigrams by Article Cluster

This section repeats the same c-TF-IDF setup using bigrams instead of unigrams.

Compute c-TF-IDF scores for bigrams.

In [77]:
bigram_vectorizer = CountVectorizer(
    lowercase=True,
    strip_accents='unicode',
    stop_words=stop_words,
    ngram_range=(2, 2),
    min_df=2,
    max_df=0.75,
    token_pattern=r'(?u)\b[a-zA-Z]{3,}\b'
)

bigram_count_matrix = bigram_vectorizer.fit_transform(cluster_docs['cluster_text'])
bigram_terms = bigram_vectorizer.get_feature_names_out()

bigram_term_counts = bigram_count_matrix.toarray().astype(float)
bigram_cluster_lengths = bigram_term_counts.sum(axis=1, keepdims=True)
bigram_tf = np.divide(
    bigram_term_counts,
    bigram_cluster_lengths,
    where=bigram_cluster_lengths != 0
)

bigram_term_presence = (bigram_term_counts > 0).sum(axis=0)
bigram_idf = np.log((1 + len(cluster_docs)) / (1 + bigram_term_presence)) + 1
bigram_ctfidf = bigram_tf * bigram_idf

bigram_ctfidf.shape

/var/folders/bw/1dh_ldh12pz9s_1_677m4czw0000gn/T/ipykernel_79786/1991946893.py:16: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  bigram_tf = np.divide(


(5, 50938)

Select the top 10 distinctive bigrams per cluster.

In [78]:
bigram_top_n = 10
bigram_rows = []

for row_idx, cluster_id in enumerate(cluster_docs['article_cluster']):
    top_indices = []

    for term_idx in np.argsort(bigram_ctfidf[row_idx])[::-1]:
        if bigram_terms[term_idx] in blocked_phrases:
            continue

        top_indices.append(term_idx)

        if len(top_indices) == bigram_top_n:
            break

    for term_idx in top_indices:
        bigram_rows.append({
            'cluster': f'Cluster {cluster_id}: {cluster_names.get(cluster_id, "Unnamed")}',
            'term': bigram_terms[term_idx],
            'ctfidf_score': bigram_ctfidf[row_idx, term_idx]
        })

bigram_top_terms = pd.DataFrame(bigram_rows)
bigram_top_terms.head(20)

,cluster,term,ctfidf_score
0,Cluster 0: Mainstream Consensus,southern lebanon,0.001896
1,Cluster 0: Mainstream Consensus,health ministry,0.001674
2,Cluster 0: Mainstream Consensus,million barrels,0.001533
3,Cluster 0: Mainstream Consensus,air base,0.001512
4,Cluster 0: Mainstream Consensus,fifth world,0.001433
5,Cluster 0: Mainstream Consensus,south pars,0.001371
6,Cluster 0: Mainstream Consensus,militant group,0.001250
7,Cluster 0: Mainstream Consensus,red crescent,0.001069
8,Cluster 0: Mainstream Consensus,brent crude,0.001049
9,Cluster 0: Mainstream Consensus,backed hezbollah,0.000928


Display separate bar charts for the most distinctive bigrams in each cluster.

In [79]:
bigram_cluster_figures = {
    cluster_label: make_cluster_word_chart(cluster_label, bigram_top_terms)
    for cluster_label in cluster_colors
}

for fig in bigram_cluster_figures.values():
    fig.show()